In [18]:
from clocks.epigenetic_clocks import Horvath, Hannum, PhenoAge, GrimAgeV2, GrimAgeAdapter, run_all_clocks
import pandas as pd
import pickle

dataset = {}
chron_ages = {}
female = {}          # ← now a dict, not a single Series

for data in ['TCGA-BRCA' , 'TCGA-LGG' , 'GSE41037']:
    with open(f'/work/gr-fe/bryan/EpiHydra/data/02_processed/{data}/methylation.pkl', 'rb') as file:
        methylation = pickle.load(file)
        
    dataset[data] = methylation['CpG'].fillna(methylation['CpG'].median())
    chron_ages[data] = methylation['meta'].set_index('ID')['age']
    female[data] = methylation['meta'].set_index('ID')['gender'] == 'female'   # ← keyed by dataset

clocks = [
    Horvath(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Horvath/horvath.csv"),
    Hannum(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Hannum/hannum.csv", intercept=0.0),
    PhenoAge(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/PhenoAge/phenoage.csv"),
    GrimAgeAdapter(                                                              
        GrimAgeV2(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/GrimAge/GrimAgeV2.csv"),
        chron_ages=chron_ages,
        female=female,
    ),
]

results = run_all_clocks(
    datasets=dataset,
    save_path="/work/gr-fe/bryan/EpiHydra/results",
    clocks=clocks,
    chron_ages=chron_ages,
)

Loaded 353 CpG coefficients + intercept.
Loaded 71 CpG coefficients.
Loaded 513 CpG coefficients + intercept.
Loaded GrimAge V2: 10 sub-components, 1030 unique CpG probes.

  Clocks   : horvath_age, hannum_age, phenoage_age, grimageadapter_age
  Datasets : TCGA-BRCA, TCGA-LGG, GSE41037

  Dataset : TCGA-BRCA
  Samples : 1097   CpGs : 422790

  → Running horvath_age (Horvath) …


/tmp/4881159/ipykernel_696164/1662948953.py:19: UserWarning: Hannum intercept is set to 0.0 (the default). This is almost certainly incorrect and will bias every prediction. Supply the correct value from the paper's supplementary materials.
  Hannum(coefficients_path="/work/gr-fe/bryan/EpiHydra/data/03_final/Hannum/hannum.csv", intercept=0.0),


⚠  8 / 353 clock CpGs (2.3%) not found in methylation data.

  → Running hannum_age (Hannum) …
⚠  5 / 71 clock CpGs (7.0%) not found in methylation data.

  → Running phenoage_age (PhenoAge) …
⚠  14 / 513 clock CpGs (2.7%) not found in methylation data.

  → Running grimageadapter_age (GrimAgeAdapter) …
  → Predicting DNAmGDF15 …
⚠  [DNAmGDF15] 16 / 137 CpGs (11.7%) missing from methylation data.
  → Predicting DNAmB2M …
⚠  [DNAmB2M] 14 / 91 CpGs (15.4%) missing from methylation data.
  → Predicting DNAmCystatinC …
⚠  [DNAmCystatinC] 16 / 87 CpGs (18.4%) missing from methylation data.
  → Predicting DNAmTIMP1 …
⚠  [DNAmTIMP1] 2 / 42 CpGs (4.8%) missing from methylation data.
  → Predicting DNAmADM …
⚠  [DNAmADM] 27 / 186 CpGs (14.5%) missing from methylation data.
  → Predicting DNAmPAI1 …
⚠  [DNAmPAI1] 42 / 211 CpGs (19.9%) missing from methylation data.
  → Predicting DNAmLeptin …
⚠  [DNAmLeptin] 33 / 187 CpGs (17.6%) missing from methylation data.
  → Predicting DNAmPACKYRS …
⚠  [DN

In [21]:
replace = True

for data in ['TCGA-BRCA' , 'TCGA-LGG' , 'GSE41037']:
    
    filepath = f'/work/gr-fe/bryan/EpiHydra/data/02_processed/{data}/methylation.pkl'
    
    with open(filepath, 'rb') as file:
        methylation = pickle.load(file)

    dataset[data] = methylation['CpG']
    chron_ages[data] = methylation['meta'].set_index('ID')['age']

    # Only merge columns that don't already exist in meta
    if replace : 
        overlap = results[data].columns.intersection(methylation['meta'].columns)
        methylation['meta'] = methylation['meta'].drop(columns=overlap)
        
    new_cols = [col for col in results[data].columns if col not in methylation['meta'].columns]
    
    if new_cols:
        methylation['meta'] = pd.merge(
            methylation['meta'],
            results[data][new_cols],
            left_on='ID',
            right_index=True,
            how='left'
        )

    with open(filepath, 'wb') as file:
        pickle.dump(methylation, file)
